# Fig. 6c / 6d 选区工作台

一个 notebook 走完:**候选样本 → 全景选区 → 窗口预览 → 点样式微调 → 逐面板矢量导出**。

要画几个面板就在 §4 的 `PANELS` 里写几条,每条自己指定 `focus_cn`。
导出是逐面板矢量(PDF/SVG),拿去 Illustrator 组版 —— 不出拼版。


In [ ]:
%matplotlib inline
import sys, importlib

sys.path.insert(0, "/data/home/chenwenhui/fig6/scripts/04_figure_scripts")
import matplotlib.pyplot as plt
import pandas as pd
import export_fig6c_e_spatial_regions as P

importlib.reload(P)
m = P._wp31b()

# ---- 全局点样式(所有面板共用;单个面板可在 PANELS 里加 style=... 覆盖)----
STYLE = {
    "point_size": 1.2,  # 基准点径
    "highlight_size_scale": 1.6,  # 重点细胞 = point_size * 2.0
    "other_size_scale": 0.6,  # 背景细胞 = point_size * 0.8
    "highlight_alpha": 0.8,
    "other_alpha": 0.4,
    "other_keep_color": True,  # 背景保留各自类别颜色(只缩小压淡)
}
SCALEBAR_UM = 100  # 面板右下角比例尺;None = 不画

# ---- 图面显示名(只影响 legend/标注,不动数据列)----
# 想改叫法直接改 P.DISPLAY_NAMES,下面所有 legend 都跟着变
pd.DataFrame(
    {
        "data value": list(P.DISPLAY_NAMES),
        "shown as": list(P.DISPLAY_NAMES.values()),
        "color": [m.CELLTYPE_COLORS.get(k, "") for k in P.DISPLAY_NAMES],
    }
)


## 1. 候选样本速查

`recommend_crop_samples.py` 从 36 个 pre 样本的冻结指标里排的。`caveat` 标出缺 SMA/CD3/PDL1 的样本。


In [ ]:
rec6c = pd.read_csv("panel_crop_candidates/fig6c_sample_recommendations.csv")
rec6d = pd.read_csv("panel_crop_candidates/fig6d_sample_recommendations.csv")

display(
    rec6c[
        [
            "role",
            "sample",
            "cn5_prop",
            "cn5_lcc_fraction",
            "cn_participant_cells",
            "caveat",
            "in_use",
        ]
    ].round(3)
)
display(
    rec6d[
        [
            "role",
            "sample",
            "cn7_prop",
            "coupling_z_CN1_CN7",
            "coupling_z_CN3_CN7",
            "coupling_z_CN4_CN7",
            "caveat",
            "in_use",
        ]
    ].round(2)
)


## 2. 全景选区

`FOCUS_CN` 决定给哪些 CN 上色、其余压灰 —— 挑区域时这个最好用。改完重跑本 cell。

| 目标 | 建议 FOCUS_CN |
|---|---|
| 6c 淋巴样/TLS | `['CN5']` |
| 6d CN1–CN7 耦合 | `['CN1','CN7']` |
| 6d CN3–CN7 耦合 | `['CN3','CN7']` |
| 6d CN4–CN7 耦合 | `['CN4','CN7']` |
| 看全部 | `None` |


In [ ]:
SAMPLE = "pt35-pre"
FOCUS_CN = ["CN5"]

fig, ax = P.overview(SAMPLE, by="CN_k10", highlight=FOCUS_CN, save=False)
fig.set_size_inches(13, 10)

## 3. 单窗口预览

从上图网格读 `[xmin, xmax, ymin, ymax]`(px,1 px = 0.5 µm)。
参考边长:6c 1250 px ≈ 625 µm,6d 3000 px ≈ 1500 µm。

**重点细胞由 `FOCUS_CN` 决定**,phenotype 和 CN 两行放大的是同一批细胞;
所有类别都保留自己的颜色,非 focus 的只是缩小压淡。


In [ ]:
ROI = [12500, 15500, 15000, 18000]

# 两行的重点细胞都由 FOCUS_CN 决定:
#   phenotype 行 = 落在这些 CN 里的细胞被放大,颜色仍按 celltype
#              -> 读出来是「这个 niche 由什么细胞组成」
#   CN 行       = 同一批细胞被放大,颜色按 CN
# 不筛 celltype:所有类型都上自己的色(cell_highlights=None)
fig, axes = P.preview_crop(
    SAMPLE,
    ROI,
    focus_cn=FOCUS_CN,
    point_size=STYLE["point_size"],
    other_keep_color=STYLE["other_keep_color"],
    style=STYLE,
    save=False,
)


### 3b. 点样式微调(可选)

同一窗口并排比几组参数,定下来再写回上面的 `STYLE`。


In [ ]:
sub, crop = P._crop_subset(SAMPLE, ROI)

TRIALS = {
    "current": STYLE,
    "bigger focus": {**STYLE, "highlight_size_scale": 2.8, "other_alpha": 0.20},
    "grey bg": {**STYLE, "other_keep_color": False},
}
fig, axes = plt.subplots(2, len(TRIALS), figsize=(4.4 * len(TRIALS), 8.8))
for col, (name, st) in enumerate(TRIALS.items()):
    for row, (cc, pal) in enumerate(
        [("celltype", m.CELLTYPE_COLORS), ("CN_k10", m.CN_COLORS)]
    ):
        m.scatter_crop(
            axes[row, col],
            sub,
            color_col=cc,
            highlights=None,
            palette=pal,
            crop=crop,
            point_size=st["point_size"],
            style=st,
            other_keep_color=st["other_keep_color"],
            emphasis_col="CN_k10",
            emphasis_values=set(FOCUS_CN),
        )
    axes[0, col].set_title(name, fontsize=9)
axes[0, 0].set_ylabel("phenotype")
axes[1, 0].set_ylabel("CN")
fig.tight_layout()


In [ ]:
SAMPLE = "pt63-pre"
FOCUS_CN = ["CN1", "CN7"]

fig, ax = P.overview(SAMPLE, by="CN_k10", highlight=FOCUS_CN, save=False)
fig.set_size_inches(13, 10)

## 4. 面板清单 —— 要画几个就写几条

每条一个面板,`focus_cn` 各自独立;`style` 可选,不写就用全局 `STYLE`。
CN 配色是模块级常量,**同一 CN 在所有面板里颜色一致**。


In [ ]:
# 要画几个面板就写几条。focus_cn 是每个面板唯一的 focus 依据 ——
# phenotype 和 CN 两行放大的都是落在这些 CN 里的细胞。
PANELS = [
    dict(
        name="6c_CR_1",
        sample="pt35-pre",
        roi=[12500, 15500, 15000, 18000],
        focus_cn=["CN5"],
    ),
    dict(
        name="6d_CR_1",
        sample="pt63-pre",
        roi=[28473, 31473, 10406, 13406],
        focus_cn=["CN1", "CN7"],
    ),
    dict(
        name="6d_NR_1",
        sample="pt50-pre",
        roi=[6168, 9168, 8103, 11103],
        focus_cn=["CN1", "CN7"],
    ),
    dict(
        name="6d_NR_2",
        sample="pt34-pre",
        roi=[18822, 21822, 38775, 41775],
        focus_cn=["CN3", "CN7"],
    ),
    dict(
        name="6d_CR_2",
        sample="pt19-pre",
        roi=[19099, 22099, 21562, 24562],
        focus_cn=["CN3", "CN7"],
    ),
    dict(
        name="6d_CR_3",
        sample="pt42-pre",
        roi=[15606, 18606, 16871, 19871],
        focus_cn=["CN4", "CN7"],
    ),
    dict(
        name="6d_NR_3",
        sample="pt31-pre",
        roi=[21924, 24924, 32704, 35704],
        focus_cn=["CN4", "CN7"],
    ),
]

pd.DataFrame([{k: v for k, v in p.items() if k != "style"} for p in PANELS])


### 4b. 给还没定窗口的面板看全景

跑一遍,把 `roi=None` 的面板逐个出全景(按各自的 `focus_cn` 上色),读完坐标回去填。


In [ ]:
for p in PANELS:
    if p["roi"] is not None:
        continue
    fig, ax = P.overview(p["sample"], highlight=p["focus_cn"], save=False)
    fig.set_size_inches(11, 8.5)
    ax.set_title(f"{p['name']}  {p['sample']}  focus={p['focus_cn']}", fontsize=11)
    plt.show()


### 4c. 批量预览已定窗口


In [ ]:
for p in PANELS:
    if p["roi"] is None:
        print(f"[skip] {p['name']} 还没填 roi")
        continue
    st = {**STYLE, **p.get("style", {})}
    print(f"--- {p['name']}  {p['sample']}  focus_cn={p['focus_cn']}")
    P.preview_crop(
        p["sample"],
        p["roi"],
        focus_cn=p["focus_cn"],
        point_size=st["point_size"],
        style=st,
        other_keep_color=st["other_keep_color"],
        save=False,
    )
    plt.show()


## 5. 逐面板矢量导出(给 Illustrator)

每个面板出 `*_phenotype.pdf/.svg`、`*_CN.pdf/.svg`、`*_legend.pdf`。

**文字全部是可编辑矢量**:比例尺标签和 legend 用 `pdf.fonttype=42` / `svg.fonttype='none'`,
在 AI 里是真文字不是轮廓;只有散点被 `rasterized=True` 内嵌成 600 dpi 位图。
实测面板 PDF = 1 个文字算子 + 2 张内嵌位图 ≈ 210 KB,legend PDF = 25 个文字算子 + 0 位图 ≈ 14 KB。


In [ ]:
EXPORT_DIR = "region_previews/ai_export"
PANEL_SIZE_IN = 1.6  # 面板成品边长(英寸),按最终版式定

written = {}
for p in PANELS:
    if p["roi"] is None:
        print(f"[skip] {p['name']} 还没填 roi")
        continue
    st = {**STYLE, **p.get("style", {})}
    written[p["name"]] = P.export_crop_panels(
        p["sample"],
        p["roi"],
        focus_cn=p["focus_cn"],
        out_stem=p["name"],
        point_size=st["point_size"],
        style=st,
        other_keep_color=st["other_keep_color"],
        panel_size_in=PANEL_SIZE_IN,
        out_dir=EXPORT_DIR,
        legend_scope="all",  # 完整类别表,多面板共用一份 legend
        cn_niche_names=False,  # True = CN legend 带上功能 niche 名
        scalebar_um=SCALEBAR_UM,
    )


## 6. 记录选择

把定稿写进 CSV 存档,同时打印可贴进 `fig6_assembly.ipynb` 的 CONFIG 片段(如果还想走拼版的话)。


In [ ]:
rows = [
    dict(
        name=p["name"],
        sample=p["sample"],
        xmin=p["roi"][0],
        xmax=p["roi"][1],
        ymin=p["roi"][2],
        ymax=p["roi"][3],
        focus_cn=";".join(p["focus_cn"]),
    )
    for p in PANELS
    if p["roi"] is not None
]
picks = pd.DataFrame(rows)
picks.to_csv("panel_crop_candidates/final_panel_picks.csv", index=False)
display(picks)

for tag in ["6c", "6d"]:
    sel = [p for p in PANELS if p["name"].startswith(tag) and p["roi"] is not None]
    if not sel:
        continue
    print(f'"panel_{tag}_samples": {[p["sample"] for p in sel]},')
    print(
        f'"crop_overrides": {{"{tag}": '
        + repr({p["sample"]: p["roi"] for p in sel})
        + "},"
    )
